# ICR Probe

Measure how closely each layer's hidden-state update follows attention, then fit a small classifier. The experiment adapts the method to token-level code defect detection. [Paper](https://arxiv.org/abs/2507.16488)

Train an independent probe on each benchmark. Line and sample scores use the maximum annotated token score. F1 thresholds come from validation.

The estimator trains for 15 epochs by default. Each epoch reports token-weighted training BCE, validation BCE, and validation token PR-AUC and ROC-AUC. Held-out metrics use the final epoch.

In [1]:
from pathlib import Path
import sys

if Path("../src/traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import logging

import pandas as pd
import torch
from IPython.display import display

from datasets.benchmarks import DATASETS, PILOT_LIMITS
from src.evaluation import TraceStore, evaluate
from src.methods import DEFAULT_EPOCHS, ICRProbe

logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("src.methods").setLevel(logging.INFO)

epochs = DEFAULT_EPOCHS
pilot = False
limits = PILOT_LIMITS if pilot else None
torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)

In [3]:
results = []
history = []
for benchmark in DATASETS:
    print(f"Training on {benchmark}", flush=True)
    splits = store.load_splits(benchmark, limits)
    estimator = ICRProbe(epochs=epochs)
    results.extend(
        {"benchmark": benchmark, **row} for row in evaluate(estimator, splits)
    )
    history.extend(
        {"benchmark": benchmark, **row} for row in estimator.training_history
    )

metrics = pd.DataFrame(results).set_index(["benchmark", "level"])
display(metrics.round(4))

Training on mbpp


ICRProbe epoch 1/15 train_loss=0.5723 validation_loss=0.4505 validation_token_pr_auc=0.1180 validation_token_roc_auc=0.3725


ICRProbe epoch 2/15 train_loss=0.5251 validation_loss=0.4535 validation_token_pr_auc=0.1185 validation_token_roc_auc=0.3739


ICRProbe epoch 3/15 train_loss=0.5248 validation_loss=0.4466 validation_token_pr_auc=0.1189 validation_token_roc_auc=0.3753


ICRProbe epoch 4/15 train_loss=0.5241 validation_loss=0.4590 validation_token_pr_auc=0.1191 validation_token_roc_auc=0.3765


ICRProbe epoch 5/15 train_loss=0.5242 validation_loss=0.4420 validation_token_pr_auc=0.1196 validation_token_roc_auc=0.3779


... (55 outputs left out)

ICRProbe epoch 13/15 train_loss=0.6398 validation_loss=0.6753 validation_token_pr_auc=0.4953 validation_token_roc_auc=0.6741


ICRProbe epoch 14/15 train_loss=0.6430 validation_loss=0.6133 validation_token_pr_auc=0.4976 validation_token_roc_auc=0.6775


ICRProbe epoch 15/15 train_loss=0.6486 validation_loss=0.6089 validation_token_pr_auc=0.4990 validation_token_roc_auc=0.6793


n  positives  positive_rate  pr_auc  \
benchmark      level                                            
mbpp           token   3373        660         0.1957  0.1877   
               line     509         54         0.1061  0.1614   
               sample    74         31         0.4189  0.3963   
humaneval      token   3714        665         0.1791  0.3849   
               line     484         55         0.1136  0.3306   
               sample    46         19         0.4130  0.3541   
humaneval_java token   1157        205         0.1772  0.1747   
               line     155         15         0.0968  0.1393   
               sample    36         11         0.3056  0.2507   
defects4j      token   3901       2220         0.5691  0.7635   
               line     448        148         0.3304  0.5941   
               sample    86         35         0.4070  0.5210   

                       average_precision  roc_auc  precision  recall      f1  \
benchmark      level                                                           
mbpp           token              0.1887   0.4666     0.1996  0.9818  0.3318   
               line               0.1720   0.6331     0.1905  0.0741  0.1067   
               sample             0.4171   0.4880     0.4189  1.0000  0.5905   
humaneval      token              0.3857   0.7096     0.3665  0.5925  0.4529   
               line               0.3358   0.6712     0.3704  0.3636  0.3670   
               sample             0.3693   0.3119     0.2692  0.3684  0.3111   
humaneval_java token              0.1763   0.5239     0.1766  0.5073  0.2620   
               line               0.1517   0.6864     0.1429  0.2667  0.1860   
               sample             0.2905   0.3982     0.3235  1.0000  0.4889   
defects4j      token              0.7638   0.7605     0.6714  0.9122  0.7735   
               line               0.5969   0.8016     0.5708  0.8986  0.6982   
               sample             0.5287   0.5770     0.4133  0.8857  0.5636   

                       threshold  
benchmark      level              
mbpp           token      0.2192  
               line       0.2306  
               sample     0.2266  
humaneval      token      0.1492  
               line       0.1529  
               sample     0.1505  
humaneval_java token      0.1787  
               line       0.1821  
               sample     0.1806  
defects4j      token      0.2892  
               line       0.3919  
               sample     0.4801

### Training history

Loss is binary cross-entropy on labeled tokens.

In [4]:
with pd.option_context("display.max_rows", None):
    display(pd.DataFrame(history).set_index(["benchmark", "epoch"]).round(4))

train_loss  validation_loss  validation_token_pr_auc  \
benchmark      epoch                                                         
mbpp           1          0.5723           0.4505                   0.1180   
               2          0.5251           0.4535                   0.1185   
               3          0.5248           0.4466                   0.1189   
               4          0.5241           0.4590                   0.1191   
               5          0.5242           0.4420                   0.1196   
               6          0.5241           0.4488                   0.1201   
               7          0.5238           0.4450                   0.1206   
               8          0.5240           0.4480                   0.1211   
               9          0.5241           0.4518                   0.1218   
               10         0.5242           0.4493                   0.1224   
               11         0.5240           0.4509                   0.1230   
               12         0.5235           0.4515                   0.1239   
               13         0.5238           0.4509                   0.1247   
               14         0.5238           0.4472                   0.1254   
               15         0.5235           0.4502                   0.1266   
humaneval      1          0.5229           0.6680                   0.4359   
               2          0.4098           0.6090                   0.4338   
               3          0.4050           0.6002                   0.4323   
               4          0.4048           0.5972                   0.4303   
               5          0.4049           0.5875                   0.4283   
               6          0.4053           0.5953                   0.4262   
               7          0.4053           0.5906                   0.4232   
               8          0.4048           0.5903                   0.4213   
               9          0.4048           0.5854                   0.4186   
               10         0.4050           0.5993                   0.4169   
               11         0.4059           0.5926                   0.4147   
               12         0.4056           0.5839                   0.4122   
               13         0.4055           0.5919                   0.4102   
               14         0.4054           0.5969                   0.4080   
               15         0.4056           0.5919                   0.4055   
humaneval_java 1          0.6288           0.5144                   0.1383   
               2          0.4906           0.3475                   0.1472   
               3          0.4695           0.3681                   0.1450   
               4          0.4648           0.3629                   0.1455   
               5          0.4647           0.3614                   0.1461   
               6          0.4646           0.3651                   0.1466   
               7          0.4642           0.3639                   0.1474   
               8          0.4640           0.3618                   0.1483   
               9          0.4641           0.3614                   0.1490   
               10         0.4643           0.3656                   0.1496   
               11         0.4641           0.3657                   0.1502   
               12         0.4639           0.3617                   0.1512   
               13         0.4645           0.3679                   0.1520   
               14         0.4646           0.3625                   0.1530   
               15         0.4639           0.3642                   0.1535   
defects4j      1          0.6927           0.6838                   0.4722   
               2          0.6925           0.6884                   0.4780   
               3          0.6923           0.6849                   0.4860   
               4          0.6921           0.6787                   0.4911   
               5          0.6918           0